In [1]:
# Mount Drive and load 70/10/20 models
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

STGCN_DIR  = '/content/st-gcn'
CTRGCN_DIR = '/content/CTR-GCN'

# Clone ST-GCN
if not os.path.exists(STGCN_DIR):
    os.system(f'git clone https://github.com/yysijie/st-gcn.git {STGCN_DIR}')
stgcn_io = Path(f'{STGCN_DIR}/torchlight/torchlight/io.py')
text = stgcn_io.read_text()
if 'weights_only=False' not in text:
    text = text.replace('torch.load(weights_path)', 'torch.load(weights_path, weights_only=False, map_location="cpu")')
    stgcn_io.write_text(text)
os.system(f'pip install -e {STGCN_DIR}/torchlight -q')

# Clone CTR-GCN
if not os.path.exists(CTRGCN_DIR):
    os.system(f'git clone https://github.com/Uason-Chen/CTR-GCN.git {CTRGCN_DIR}')
os.system('pip install -q tensorboardX torchpack fvcore iopath yacs thop')
os.system(f'pip install -e {CTRGCN_DIR}/torchlight -q')
ctrgcn_util = Path(f'{CTRGCN_DIR}/torchlight/torchlight/util.py')
text = ctrgcn_util.read_text()
if 'PaviLogger = None' not in text:
    text = text.replace('from torchpack.runner.hooks import PaviLogger', 'try:\n    from torchpack.runner.hooks import PaviLogger\nexcept ImportError:\n    PaviLogger = None')
    ctrgcn_util.write_text(text)

print("Loading model architectures...")

# Load ST-GCN
sys.path.insert(0, STGCN_DIR)
from net.st_gcn import Model as STGCN_Model
stgcn_model = STGCN_Model(in_channels=3, num_class=30, dropout=0.5, edge_importance_weighting=True, graph_args={'layout': 'ntu-rgb+d', 'strategy': 'spatial'})
stgcn_ckpt = torch.load('/content/drive/MyDrive/HRC_Research/checkpoints/stgcn_hri30_70_10_20/best_stgcn_hri30.pt', map_location='cpu', weights_only=False)
stgcn_state = stgcn_ckpt['model_state_dict'] if isinstance(stgcn_ckpt, dict) and 'model_state_dict' in stgcn_ckpt else stgcn_ckpt
stgcn_state = {k[7:] if k.startswith('module.') else k: v for k, v in stgcn_state.items()}
stgcn_model.load_state_dict(stgcn_state, strict=True)
stgcn_model = stgcn_model.to(device)
stgcn_model.eval()
print("ST-GCN loaded ✓")

# Load CTR-GCN
sys.path.insert(0, CTRGCN_DIR)
from model.ctrgcn import Model as CTRGCN_Model
ctrgcn_model = CTRGCN_Model(num_class=30, num_point=25, num_person=1, graph='graph.ntu_rgb_d.Graph', graph_args={'labeling_mode': 'spatial'})
ctrgcn_ckpt = torch.load('/content/drive/MyDrive/HRC_Research/checkpoints/ctrgcn_hri30_70_10_20/best_ctrgcn_hri30.pt', map_location='cpu', weights_only=False)
ctrgcn_state = ctrgcn_ckpt['model_state_dict'] if isinstance(ctrgcn_ckpt, dict) and 'model_state_dict' in ctrgcn_ckpt else ctrgcn_ckpt
ctrgcn_state = {k[7:] if k.startswith('module.') else k: v for k, v in ctrgcn_state.items()}
ctrgcn_model.load_state_dict(ctrgcn_state, strict=True)
ctrgcn_model = ctrgcn_model.to(device)
ctrgcn_model.eval()
print("CTR-GCN loaded ✓")

Mounted at /content/drive
Using device: cuda
Loading model architectures...
ST-GCN loaded ✓
CTR-GCN loaded ✓


In [2]:
# Load 70/10/20 test data
import numpy as np
import pickle

STGCN_TEST_DATA  = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/stgcn_format/test_data.npy'
STGCN_TEST_LABEL = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/stgcn_format/test_label.pkl'
CTRGCN_TEST_NPZ  = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/ctrgcn_format/HRI30_CS.npz'

stgcn_test_data = np.load(STGCN_TEST_DATA)
with open(STGCN_TEST_LABEL, 'rb') as f:
    _, y_test = pickle.load(f)
stgcn_test_labels = np.array(y_test)

npz = np.load(CTRGCN_TEST_NPZ)
ctrgcn_test_data   = npz['x_test']

print(f"ST-GCN test data shape:   {stgcn_test_data.shape}")
print(f"CTR-GCN test data shape:  {ctrgcn_test_data.shape}")
print("Test data loaded successfully.")

ST-GCN test data shape:   (588, 3, 150, 25, 1)
CTR-GCN test data shape:  (588, 3, 150, 25, 1)
Test data loaded successfully.


In [3]:
# Structured Limb Occlusion Experiment
import time
import csv

# Correct NTU RGB+D 25-joint limb groups (0-indexed)
LIMB_GROUPS = {
    "left_arm":  [4, 5, 6],   # Left Shoulder, Elbow, Wrist
    "right_arm": [8, 9, 10],  # Right Shoulder, Elbow, Wrist
    "left_leg":  [12, 13, 14],# Left Hip, Knee, Ankle
    "right_leg": [16, 17, 18] # Right Hip, Knee, Ankle
}

def structured_occlusion(data_np, limb_name, seed=None, block_len_ratio=0.3):
    """
    Zero out all joints of limb_name for a contiguous block of frames.
    block_len_ratio=0.3 means 45 frames out of 150 are zeroed.
    The start frame is randomized to simulate occlusion happening at different times.
    """
    if seed is not None:
        np.random.seed(seed)
    out = data_np.copy()
    T = out.shape[2] # T=150
    joints = LIMB_GROUPS[limb_name]
    block_len = int(T * block_len_ratio)

    # Randomly pick a start frame, ensuring the block fits within the sequence
    t_start = np.random.randint(0, T - block_len + 1)
    out[:, :, t_start:t_start+block_len, joints, :] = 0.0
    return out

def get_model_logits(model, data_np, batch_size=64):
    model.eval()
    all_logits = []
    N = data_np.shape[0]
    with torch.no_grad():
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            batch = torch.tensor(data_np[start:end], dtype=torch.float32).to(device)
            output = model(batch)
            all_logits.append(output.cpu().numpy())
    return np.concatenate(all_logits)

def softmax(logits):
    exp_x = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def entropy(probs):
    return -np.sum(probs * np.log(probs + 1e-9), axis=1)

def consensus_fusion(logits_st, logits_ctr):
    probs_st  = softmax(logits_st)
    probs_ctr = softmax(logits_ctr)
    H_st  = entropy(probs_st)
    H_ctr = entropy(probs_ctr)
    W_st  = np.exp(-H_st)
    W_ctr = np.exp(-H_ctr)
    sum_W = W_st + W_ctr
    W_st  = W_st  / sum_W
    W_ctr = W_ctr / sum_W
    probs_fused = W_st[:, None] * probs_st + W_ctr[:, None] * probs_ctr
    return probs_fused

N_TRIALS = 30
N = stgcn_test_labels.shape[0]
results = []

print("=" * 70)
print("M7: STRUCTURED LIMB OCCLUSION EXPERIMENT (N=30 TRIALS)")
print("Zeroing 1 limb for 45 contiguous frames (30% of sequence)")
print("=" * 70)

for limb in LIMB_GROUPS.keys():
    print(f"\n--- Limb: {limb} ---")
    st_accs, ctr_accs, con_accs = [], [], []
    t_start = time.time()

    for trial in range(N_TRIALS):
        seed = 42 + trial

        # Apply structured occlusion
        st_occ = structured_occlusion(stgcn_test_data, limb, seed=seed)
        ctr_occ = structured_occlusion(ctrgcn_test_data, limb, seed=seed)

        # Run inference
        logits_st = get_model_logits(stgcn_model, st_occ)
        logits_ctr = get_model_logits(ctrgcn_model, ctr_occ)

        # Predictions
        st_preds = np.argmax(logits_st, axis=1)
        ctr_preds = np.argmax(logits_ctr, axis=1)
        con_preds = np.argmax(consensus_fusion(logits_st, logits_ctr), axis=1)

        # Accuracy
        st_accs.append(100.0 * (st_preds == stgcn_test_labels).sum() / N)
        ctr_accs.append(100.0 * (ctr_preds == stgcn_test_labels).sum() / N)
        con_accs.append(100.0 * (con_preds == stgcn_test_labels).sum() / N)

    st_mean, st_std = float(np.mean(st_accs)), float(np.std(st_accs))
    ctr_mean, ctr_std = float(np.mean(ctr_accs)), float(np.std(ctr_accs))
    con_mean, con_std = float(np.mean(con_accs)), float(np.std(con_accs))

    print(f"  ST-GCN:    {st_mean:.2f} ± {st_std:.2f}%")
    print(f"  CTR-GCN:   {ctr_mean:.2f} ± {ctr_std:.2f}%")
    print(f"  Consensus: {con_mean:.2f} ± {con_std:.2f}%")
    print(f"  Time: {time.time() - t_start:.1f}s")

    results.append({
        'limb': limb,
        'st_mean': st_mean, 'st_std': st_std,
        'ctr_mean': ctr_mean, 'ctr_std': ctr_std,
        'con_mean': con_mean, 'con_std': con_std
    })

# Save to Drive
OUT_DIR = '/content/drive/MyDrive/HRC_Research/results/structured_occlusion'
os.makedirs(OUT_DIR, exist_ok=True)
csv_path = os.path.join(OUT_DIR, 'structured_occlusion_results_70_10_20.csv')

with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['limb', 'st_mean', 'st_std', 'ctr_mean', 'ctr_std', 'con_mean', 'con_std'])
    writer.writeheader()
    writer.writerows(results)

print(f"\nSaved -> {csv_path}")

M7: STRUCTURED LIMB OCCLUSION EXPERIMENT (N=30 TRIALS)
Zeroing 1 limb for 45 contiguous frames (30% of sequence)

--- Limb: left_arm ---
  ST-GCN:    52.94 ± 1.86%
  CTR-GCN:   61.33 ± 5.65%
  Consensus: 63.26 ± 4.01%
  Time: 136.9s

--- Limb: right_arm ---
  ST-GCN:    50.50 ± 4.13%
  CTR-GCN:   58.45 ± 8.94%
  Consensus: 59.95 ± 7.30%
  Time: 139.1s

--- Limb: left_leg ---
  ST-GCN:    54.34 ± 0.66%
  CTR-GCN:   58.81 ± 9.31%
  Consensus: 63.65 ± 4.66%
  Time: 139.4s

--- Limb: right_leg ---
  ST-GCN:    54.14 ± 1.23%
  CTR-GCN:   58.44 ± 9.26%
  Consensus: 64.12 ± 4.67%
  Time: 139.5s

Saved -> /content/drive/MyDrive/HRC_Research/results/structured_occlusion/structured_occlusion_results_70_10_20.csv


In [4]:
# Print Final Summary Table
import pandas as pd

print("=" * 80)
print("TABLE XII: STRUCTURED LIMB OCCLUSION RESULTS (Mean ± Std, N=30 trials)")
print("=" * 80)
print(f"{'Limb':<12} {'ST-GCN':>14} {'CTR-GCN':>14} {'Consensus':>15} {'Improvement':>13}")
print("-" * 80)

for r in results:
    improvement = r['con_mean'] - max(r['st_mean'], r['ctr_mean'])
    print(f"{r['limb']:<12} "
          f"{r['st_mean']:>6.2f}±{r['st_std']:>4.2f}  "
          f"{r['ctr_mean']:>6.2f}±{r['ctr_std']:>4.2f}  "
          f"{r['con_mean']:>7.2f}±{r['con_std']:>4.2f}  "
          f"{improvement:>+11.2f} pp")

print("=" * 80)
print("Copy this table into Section V-G of the paper.")

TABLE XII: STRUCTURED LIMB OCCLUSION RESULTS (Mean ± Std, N=30 trials)
Limb                 ST-GCN        CTR-GCN       Consensus   Improvement
--------------------------------------------------------------------------------
left_arm      52.94±1.86   61.33±5.65    63.26±4.01        +1.93 pp
right_arm     50.50±4.13   58.45±8.94    59.95±7.30        +1.51 pp
left_leg      54.34±0.66   58.81±9.31    63.65±4.66        +4.84 pp
right_leg     54.14±1.23   58.44±9.26    64.12±4.67        +5.67 pp
Copy this table into Section V-G of the paper.
